# Exact solver: time vs cores

How fast is Hila's exact solver (`rust/exact`) with more cores, and on which graphs? It solves
the Moran process on all 2^N mutant sets with Gauss-Seidel sweeps, and its threads split each
sweep between them.

Each graph gets one LSF job that takes `max(CORES)` cores on one host and solves the graph once
per core count, timing each run. Keeping a graph's runs in one job holds its host and CPU fixed,
so its times are comparable. Different graphs run as separate jobs at the same time, possibly on
different CPUs; the figure's footnote says where each one ran.

**To use:** list graphs in section 1 and run all cells. Nothing blocks or raises while jobs run:
the submit cell submits only what is missing, and the last cell shows whatever has finished.
Re-run the last cell to refresh.

**Saved per graph and r** under `BENCH_ROOT`, in folders named `<graph name>_<WL hash>_r<r>`:
the solver inputs (graph and r), each run's per-node results (`prob`, `time`, `ctime`), the
timings and the host. Graphs you keep in `GRAPHS` are read from disk, not solved again. Delete a
graph's folder to measure it again.

In [ ]:
%cd /home/labs/pilpel/matanyaw/moran-process
import json, re, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator
from moran_graphs import PopulationGraph
from moran_graphs.hpc import ALIVE, DATA_ROOT
from moran_graphs.properties import wl_hash

CRATE = Path("rust/exact").resolve()
SOLVER = CRATE / "target/release/numeric-solver"
BENCH_ROOT = DATA_ROOT / "benchmarks" / "exact-solver-cores"
# Incremental, a no-op when the binary is newer than the Rust sources (~6 s otherwise).
subprocess.run([Path.home() / ".cargo/bin/cargo", "build", "--release", "--bin", "numeric-solver"], cwd=CRATE, check=True)

## How it works

`submit(g)` writes one solver input per core count and pipes a job script to `bsub`. The job
binds `max(CORES)` cores and, for each `c` in `CORES`, runs the solver with `"threads": c`,
appending its start and end times to `times.tsv`. The other helpers only read what jobs wrote.

In [ ]:
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # graph i gets COLORS[i]


def bench_dir(g):
    # The WL hash is the graph's identity; names alone repeat (random_graph's name has no seed).
    return BENCH_ROOT / f"{g.name}_{wl_hash(g.csr())}_r{R}"


def job_name(g):
    return f"exact_cores_{bench_dir(g).name}"


def job_state(g):
    # bjobs lists only unfinished jobs; once LSF drops one, the end of its log says how it ended.
    listed = subprocess.run(["bjobs", "-noheader", "-o", "stat", "-J", job_name(g)],
                            capture_output=True, text=True).stdout.split()
    log = bench_dir(g) / "job.log"
    if listed:
        return listed[-1]
    if not log.exists():
        return "not submitted"
    return "DONE" if "Successfully completed" in log.read_text() else "EXIT"


def submit(g):
    folder, n_nodes = bench_dir(g), g.graph.number_of_nodes()
    folder.mkdir(parents=True, exist_ok=True)
    adjacency = [sorted(g.graph.neighbors(node)) for node in range(n_nodes)]
    for cores in CORES:
        line = json.dumps({"graph": adjacency, "fitness": R, "threads": cores})
        (folder / f"input_c{cores}.json").write_text(line + "\n")
    memory_mb = 12 * 2**n_nodes / 2**20 * 1.2 + 512  # three f32 per state, plus headroom
    per_core_mb = int(memory_mb / max(CORES)) + 1  # WEXAC reserves rusage per core
    script = f"""#!/bin/bash
#BSUB -J {job_name(g)}
#BSUB -q {QUEUE}
#BSUB -n {max(CORES)}
#BSUB -R "span[hosts=1] rusage[mem={per_core_mb}]"
#BSUB -oo {folder}/job.log
set -euo pipefail
cd {folder}
rm -f times.tsv output_c*.json  # a rerun starts clean, so times.tsv never mixes two runs
echo "ran on $(hostname): $(nproc) cores bound, $(lscpu | sed -n 's/^Model name: *//p')"
for cores in {' '.join(map(str, CORES))}; do
    start=$(date +%s.%N)
    {SOLVER} < input_c$cores.json > output_c$cores.json
    echo "$cores $start $(date +%s.%N)" >> times.tsv
done
"""
    # Script on stdin: the WEXAC job starter re-evaluates $vars and quotes on the bsub line.
    out = subprocess.run(["bsub"], input=script, text=True, capture_output=True, check=True)
    print(f"{g.name}: {out.stdout.strip()}")


def load_times(g):
    # One row per finished core count; rows appear while the job runs.
    path = bench_dir(g) / "times.tsv"
    if not path.exists():
        return pd.DataFrame()
    times = pd.read_csv(path, sep=" ", names=["cores", "start", "end"], index_col="cores")
    times["seconds"] = times.end - times.start
    times["speedup"] = times.seconds.iloc[0] / times.seconds
    # Mean fixation probability over start nodes: more threads must not change the answer.
    times["rho"] = [pd.read_json(path.parent / f"output_c{cores}.json").prob.mean() for cores in times.index]
    return times.drop(columns=["start", "end"])


def ran_on(g):
    log = bench_dir(g) / "job.log"
    found = re.search(r"^ran on .*", log.read_text(), re.M) if log.exists() else None
    return found.group(0) if found else ""


def draw_graphs(axes):
    for g, color in zip(GRAPHS, COLORS):
        g.draw(ax=axes[g.name], descriptive=False, node_color=color,
               title=f"{g.name}\nN = {g.graph.number_of_nodes()}")
        axes[g.name].title.set_fontsize(11)


def plot_benchmark(runs):
    fig, axes = plt.subplot_mosaic([[g.name for g in GRAPHS], ["time"] * len(GRAPHS)], height_ratios=[1, 1.6],
                                   figsize=(max(9, 3.5 * len(GRAPHS)), 7.5))
    draw_graphs(axes)
    ax = axes["time"]
    for g, color in zip(GRAPHS, COLORS):
        times = runs[g.name]
        if times.empty:
            continue
        ideal = times.seconds.iloc[0] * times.index[0] / np.array(CORES)
        ax.plot(CORES, ideal, "--", color=color, lw=1, alpha=0.6)
        ax.plot(times.index, times.seconds, "o-", color=color, lw=2, ms=7, label=g.name)
    ax.plot([], [], "--", color="#8a8984", lw=1, label="ideal scaling")
    ax.set(xscale="log", yscale="log", xlabel="cores", ylabel="wall time (s)")
    ax.set_xticks(CORES, CORES)
    ax.yaxis.set_major_locator(LogLocator(subs=(1, 2, 5)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda seconds, _: f"{seconds:g}"))
    ax.minorticks_off()
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    hosts = [f"{g.name} {ran_on(g)}" for g in GRAPHS if ran_on(g)]
    fig.text(0.99, 0.0, "\n".join(hosts), ha="right", va="top", fontsize=8, color="#52514e")
    return fig

## 1. Graphs

Append to `GRAPHS` and keep the earlier graphs: they cost nothing once solved, and they stay in
the figure. Up to 8 graphs, with distinct names (rename one of two `random_graph`s with
`g.name = ...`). Any factory works if the graph is undirected (directed graphs are unverified in
this solver) and has 10 to 63 nodes. The solver keeps three f32 per mutant set, so memory is
12 B x 2^N (N = 24 is 200 MB, N = 28 is 3 GB). On one thread this avian family took 2.4 s at
N = 18 (login node) and 715 s at N = 24 (Xeon Gold 5320), about 2.6x per extra node.

In [ ]:
GRAPHS = [
    PopulationGraph.avian_graph(n_parabronchi=7, l_parabronchi=3),
    PopulationGraph.mammalian_lung_graph(depth=3, branching_factor=2),
    PopulationGraph.fish_graph(n_rods=3, l_rod=2, fillaments=1),
    PopulationGraph.star_graph(15),
    PopulationGraph.complete_graph(15),
]
R = 1.1
CORES = [1, 2, 4, 8, 16]  # run in this order
QUEUE = "gsla-cpu"  # the lab queue is never preempted, so no suspension lands inside a timing

assert len({g.name for g in GRAPHS}) == len(GRAPHS), "two graphs share a name: set g.name"
assert len(GRAPHS) <= len(COLORS), f"at most {len(COLORS)} graphs, one color each"
fig, axes = plt.subplot_mosaic([[g.name for g in GRAPHS]], figsize=(max(6, 3.5 * len(GRAPHS)), 3.5))
draw_graphs(axes)
plt.show()

## 2. Submit

Safe to re-run: for each graph, it submits only when results are missing and no job for that
graph is queued or running. A failed job is left for you to read, not resubmitted.

In [ ]:
for g in GRAPHS:
    state = job_state(g)
    if set(CORES) <= set(load_times(g).index):
        print(f"{g.name}: all core counts finished")
    elif state in ALIVE:
        print(f"{g.name}: job {state}")
    elif state == "EXIT":
        print(f"{g.name}: the last job failed, see {bench_dir(g) / 'job.log'}; delete that log to resubmit")
    else:
        submit(g)

## 3. Results

Re-run while jobs run: the table and the figure fill in as each core count finishes. Dashed lines
are ideal scaling from each graph's first core count. `rho`, the mean fixation probability over
start nodes, should agree across a graph's rows to solver tolerance. Once every graph has every
core count, the figure is saved as `time_vs_cores_r<r>.png` and `.pdf` in `BENCH_ROOT`.

In [ ]:
runs = {g.name: load_times(g) for g in GRAPHS}
for g in GRAPHS:
    print(f"{g.name}: {len(runs[g.name])}/{len(CORES)} core counts finished, job {job_state(g)}")
done = {name: times for name, times in runs.items() if len(times)}
if done:
    display(pd.concat(done, names=["graph"]).round({"seconds": 1, "speedup": 2}))
    fig = plot_benchmark(runs)
    if all(set(CORES) <= set(times.index) for times in runs.values()):
        for ext in ("png", "pdf"):
            fig.savefig(BENCH_ROOT / f"time_vs_cores_r{R}.{ext}", dpi=150, bbox_inches="tight")
        print(f"saved {BENCH_ROOT}/time_vs_cores_r{R}.png and .pdf")
    plt.show()